# Quantum Circuit Compilation from Source

This notebook builds circuits for the three benchmarks of the competition: an inverse-square-root phase for Coulomb interactions in fusion plasmas, a Reed–Solomon decoder for Decoded Quantum Interferometry (DQI), and the angle lookups of a FeMoCo block encoding. It then checks each circuit with the official verifier at its full number of shots. All circuits are kickmix circuits, made of reversible classical gates, phase gates and measurement-based uncomputation, so a classical simulator can check them input by input.

The score is the geometric mean of $\text{cost}_\text{ref}/\text{cost}$ over the three tasks. The costs are defined by the organizers as follows.

| Task | Cost | Reference circuit | This notebook |
|---|---|---|---|
| Inverse square root | Q × (Toffoli + T / 4) | Q 260, 7,735 Toffolis | Q 78, 1,696 Toffoli-equivalents |
| DQI decoder | R(ε₊) × Q² × T | Q 1,893, 5.73 M Toffolis | Q 881, 0.96 M Toffolis |
| FeMoCo lookups | λ_eff × T_total × Q_total | Q 730, 93,237 Toffolis, λ_eff 49.30 | Q 833, 20,286 Toffolis, λ_eff 47.86 |

Q is the qubit count. In the first row T counts T gates, and synthesized rotations are priced in T gates. In the second row T is the mean Toffoli count per shot, and R(ε₊) grows with the decoder's measured failure rate. T_total and Q_total cover the whole block encoding, including the organizers' state-preparation costs.

Each section below follows the same order: why the subroutine matters, what the circuit computes, where the savings come from, and which approximation is used and how much of the accuracy budget it spends.

In [ ]:
import json, math, os, pathlib, sys
os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get('QCC_INPUT', '/kaggle/input'))
WORK = pathlib.Path(os.environ.get('QCC_WORK', '/kaggle/working'))
SUB = pathlib.Path(os.environ.get('QCC_SUB', '/tmp/qcc_submission'))
SLUG = 'google-verifiable-quantum-circuit-compilation'
COMP = next(p for p in [INPUT / 'competitions' / SLUG, INPUT / SLUG] if (p / 'kickmix').exists())
# The chemistry numeric input is located by the build cell.
WORKERS = int(os.environ.get('QCC_WORKERS', os.cpu_count() or 4))
for d in (SUB, WORK / 'src' / 'gen_invsqrt_r2', WORK / 'src' / 'gen_dqi_v6' / 'ref'):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
REFERENCE = {'invsqrt': 2076774.233715282, 'dqi': 24439474177519.598, 'dfthcsos': 3355599588.3155365}

for _d in []:
    (WORK / _d).mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib, shutil
DATA = next(p.parent for p in INPUT.rglob('femoco_thc_r5.npz'))
shutil.copytree(next(DATA.rglob('public_v11.py')).parent, WORK / 'src', dirs_exist_ok=True)
CHEM = DATA / 'femoco_thc_r5.npz'
with CHEM.open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == '809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5'
sys.path.insert(0, str(WORK / 'src'))
import qcc_pipeline as P
P.setup(COMP)
import public_v11 as V
V.build_invsqrt(SUB, WORK / 'build' / 'invsqrt', COMP)
V.build_dqi(SUB, WORK / 'build' / 'dqi', COMP)
V.build_dfthcsos(SUB, CHEM)
RESULTS = P.verify_all(SUB, 'notebook-verification', workers=WORKERS)
assert all(r['status'] == 'PASS' for r in RESULTS.values())
(WORK / 'verification.json').write_text(json.dumps(RESULTS, indent=1, default=str))
P.write_zip(SUB, WORK / 'submission.zip')

## 1. Inverse square root

**Role.** Stopping-power simulations of warm dense plasmas evaluate a pairwise Coulomb potential. The quantum subroutine applies the phase $e^{2\pi i/\sqrt{x}}$ to a 22-bit input $x$. Every gate in it is repeated for every particle pair and time step.

**Computation.** Binary shift stages normalize $x$ to a mantissa $M \in [1, 2)$. Their flags give the shift, with $e$ its parity and $s$ half of it:

$$\frac{1}{\sqrt{x}} = 2^{\,s-10}\, y, \qquad y = \frac{2^{(e-1)/2}}{\sqrt{M}} \in (0.5, 1].$$

A lookup addressed by $e$ and the high mantissa bits loads the coefficients of a centered quadratic. The low bits form an offset whose digits are all $\pm 1$:

$$\delta = \sum_i (2d_i - 1)\, 2^{\,i-22}, \qquad R = K_1 - C_2\,\delta, \qquad y = K_0 - R\,\delta .$$

**Savings.**
- Because every digit is $\pm 1$, each Horner row is a plain addition or subtraction rather than a controlled multiplication.
- The derivative register is split into a short residual and coarse bits, the constant high part of the result is phased early, and the coefficient registers are erased by measurement with one merged phase repair.
- The lowest accumulator bit is kept virtual and its carry is formed algebraically; the derivative's lowest wire is removed by refitting the parity of $K_1$ per segment.
- The eleven narrowest outer rows add into part of the low word plus a two-bit counter that is folded in once, and the two full-width rows start with a one-bit ripple prefix.
- Normalization uses a ladder of one-bit stages with a refitted margin.
- The output phase is fused: shifts by one and two positions are applied with controlled swaps, and each rotation reads its bit through a single temporary.
- Gates whose outcome is fixed on every legal input are replaced by Clifford operations, and eligible erasing Toffolis become an X-basis measurement followed by a conditioned CZ.\n- Phase terms that are constant on the masked input domain are dropped, rotations whose angle allows it are moved to Clifford positions, and constant curvature blocks are folded.\n- Four further qubits are removed by reusing wires whose values are known on the legal domain; at 78 qubits one qubit is worth about 22 Toffolis, so the circuit accepts a few more Toffolis to save them. A final exact-domain pass removes one more Toffoli.

**Approximation.** The quadratic coefficients use short precisions and are fitted exactly on the reachable inputs. The 44 rotation angles are calibrated over the whole input domain; the worst phase error reaches 0.9998 of the tolerance after rotation synthesis, so almost the whole budget is spent. The verifier tests every legal input 64 times.

## 2. DQI decoder

**Role.** Decoded Quantum Interferometry turns an optimization problem over a finite field into the decoding of a Reed–Solomon code. Its dominant cost is a reversible decoder that runs in superposition.

**Computation.** The decoder receives 70 syndrome symbols over GF($2^{12}$) and a classical record $c$. It applies $(-1)^{\langle c, e\rangle}$ for the error $e$ of weight at most 35 and returns all workspace to zero. The locator $\sigma$ and evaluator $\Omega$ solve

$$S(z)\,\sigma(z) \equiv \Omega(z) \pmod{z^{70}}.$$

**Savings.**
- The 70 syndrome words are rewritten in place as the partial quotients of a continued fraction of $S(z)$, so the decoder needs almost no workspace beyond its 840 input qubits.
- Consecutive levels share one power-series reciprocal, so each pair of partial quotients costs a single inversion.
- The fraction is converted in place into $\sigma$ and $\Omega$, a Chien/Forney sweep applies the phase under the record bits $c_j$, and both transformations then run backwards.
- Seventy classical field words $\lambda_i$, solved from a Vandermonde system, absorb the record at positions 0–69 into a known syndrome phase $\mathrm{Tr}(\sum_i \lambda_i s_i)$, so the sweep skips those positions.
- The extraction buffer is released right after its measurement, the Forney evaluation keeps the denominator's constant coefficient implicit, and the gap history is stored as two short pointers.
- The tower-field norm of an inverse is the inverse of the norm, so the norm word is recycled instead of stored, and pairs of field products share their matrix schedule.\n- For each extraction step the circuit chooses between single, paired and matrix-paired reciprocals. The choice comes from a dynamic program over the measured Toffoli and qubit cost of every option, and the conditional scaler is reused between pairs.

**Approximation.** A missing leading coefficient is repaired by a paired 3+1 step, and every history with at most two gaps is represented exactly. Rarer patterns are not handled. On the competition distribution they occur too seldom to raise $R(\varepsilon_+)$ above its floor, which the verifier sets from a Clopper–Pearson bound over 102,400 shots.

## 3. FeMoCo angle lookups

**Role.** FeMoCo is the active site of nitrogenase. Its block encoding is the core of a quantum chemistry algorithm for nitrogen fixation. The lookup circuits stream the 75 Givens-rotation angles of every factor vector into word registers.

**Factorization.** The two-body tensor is written with one shared group of 496 unit vectors and 384 weight columns, $(R, B, C) = (1, 496, 384)$. Only 572 vectors are addressed: 76 one-body and 496 two-body, instead of 3,976 in the public 65-group refinement of the organizers' baseline. The factors were trained in four steps:
1. a fit to the shifted two-body tensor with a penalty on the non-semidefinite part of the weights;
2. quantization to 11-bit angles, in order of each angle's effect on the error;
3. word-by-word integer repair;
4. a coupled refit of weights and identity offsets against the complete Hamiltonian, with the two-body words fixed, refreshed one-body rows and an orthogonal rotation of the weight columns.

The numeric arrays are in the attached dataset, and the circuits are rebuilt from them here.

**Circuits.** Each vector is encoded on the organizers' log-depth tree with signed subtree norms, so every angle lies in $[-\pi/2, \pi/2)$. Ten words are loaded per circuit, which gives eight circuits. Each lookup table is written in algebraic normal form: every output bit is an XOR of AND-monomials of the address bits. Monomials that cannot affect a legal address are dropped, and the AND products are shared and erased by measurement. Every circuit fits into 128 qubits.

**Approximation.** The factorization spends the accuracy budget almost fully: the two-body error is 19.9995 mHa and the one-body error 6.38 mHa, against limits of 20 mHa. The normalization $\lambda_\text{eff} = \sqrt{E_\text{gap}(2\Lambda - E_\text{gap})}$ is pushed down by placing $E_\text{gap}$ at its required minimum of 3.914898 Ha.

## 4. Results

All values come from the official verifiers at full shot counts in this run. The decoder's failure count and the mean Toffoli counts depend slightly on the verification seed.

In [ ]:
r = RESULTS
rows = [
    ('Inverse square root', r['invsqrt']['num_qubits'], f"{r['invsqrt']['toffoli_count']:,.0f}",
     f"C = {r['invsqrt']['toffoli_equivalent']:,.0f} ({r['invsqrt']['rotation_count']:.0f} rotations)", 'invsqrt'),
    ('DQI decoder', r['dqi']['num_qubits'], f"{r['dqi']['toffoli_count']:,.0f}",
     f"R = {r['dqi']['repetition_factor']:.3f} ({r['dqi']['value_errors'] + r['dqi']['phase_errors']} failed of {r['dqi']['verified_shots']:,})", 'dqi'),
    ('FeMoCo lookups', r['dfthcsos']['num_qubits'], f"{r['dfthcsos']['toffoli_count']:,.0f}",
     f"λ_eff = {r['dfthcsos']['lambda_eff']:.2f}", 'dfthcsos'),
]
table = pd.DataFrame([{'Task': name, 'Qubits': q, 'Toffoli': t, 'Other': other,
                       'Cost': f"{r[k]['score']:.4g}", 'Reference cost': f'{REFERENCE[k]:.4g}',
                       'Reference / cost': f"{REFERENCE[k] / r[k]['score']:.2f}"} for name, q, t, other, k in rows])
score = math.prod(REFERENCE[k] / r[k]['score'] for k in REFERENCE) ** (1 / 3)
display(table.style.hide(axis='index'))
print(f'Score (geometric mean of reference / cost): {score:.3f}')

## Sources and reproducibility

The notebook runs on a CPU with the competition data and the attached dataset. The generator sources are in the dataset folder `circuit_sources/`; the code cell copies them, builds all three circuits, runs the official verifiers and writes `submission.zip` only after all three pass.

- Organizers: task definitions, verifiers, kickmix and the log-depth Givens tree ([competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation), [arXiv:2610.23832](https://arxiv.org/abs/2610.23832)).
- Inverse square root: the circuit descends from Phuc Thai's EXP-222/226a line ([dataset](https://www.kaggle.com/datasets/phucthaiv02/qcc-best-invsqrt-circuit)). Its construction extends the 84-qubit circuit of this notebook's earlier versions, which builds on [Sumit Chongder](https://www.kaggle.com/code/sumitchongder/notebook2f196e646b) (signed digits), [Rudra](https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing) (derivative split), [Artem1410](https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation) and [kozykappa](https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation) (exact-domain rewrites), [zongaobian](https://www.kaggle.com/code/zongaobian/quantum-circuits-signed-digits-bm-psi-qrom) (virtual low bit) and [lightningv08](https://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra) / [Mizero](https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char) (calibration). Phuc Thai added the carry-save rows, the parity refit, the normalization ladder, the measurement-plus-CZ rewrite and the domain phase mask (EXP-226a); the fused phase stage is due to [Marsyas](https://www.kaggle.com/code/marsyas/vqcc-invsqrt-13-94x-full-source). The 78-qubit circuit used here is [busyaprime's](https://www.kaggle.com/code/busyaprime/33-lb-11-94676-78-qubit-invsqrt-all-verified) seven verified edits of Phuc Thai's EXP222 ([dataset](https://www.kaggle.com/datasets/busyaprime/qcc33-verified-circuits), Apache 2.0); the last exact-domain pass is ours.
- DQI: in-place continued fraction, gap repair and arithmetic by [somejoint](https://www.kaggle.com/code/somejoint/quantum-circuits-8-95-lb-cost-breakdown); classical-mask projection by [manwithacat](https://www.kaggle.com/code/manwithacat/dqi-classical-mask-projection-9-13909); paired extraction by [Phuc Thai](https://www.kaggle.com/code/phucthaiv02/qcc-verified-portfolio-df-thc-retraining-study) and [ahmedberatozer](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression); norm recycling and paired matrix schedule by [anazemcev](https://www.kaggle.com/code/anazemcev/quantum-circuits-thc-joint-norm-881q-dqi). The decoder base is somejoint's source; the cost-driven extraction schedule and the shared scaler are ours. The reversible decoder line goes back to A. Gu et al., [arXiv:2610.22367](https://arxiv.org/abs/2610.22367).
- FeMoCo: the single-group factorization and its algebraic-normal-form compiler are ours; the penalized fit and sensitivity-ordered quantization follow ahmedberatozer, the complete-Hamiltonian joint refit follows anazemcev, the lookup and erasure lineage credits Artem1410 and kozykappa, and banked lookups go back to Sumit Chongder.
- C. Gidney, [Halving the cost of quantum addition](https://quantum-journal.org/papers/q-2018-06-18-74/) (2018); G. H. Low et al., [PRX 15, 041016](https://journals.aps.org/prx/abstract/10.1103/pb2g-j9cw) (2025).

All reused notebook code is Apache 2.0 (Kaggle's licence for public notebooks unless stated otherwise); this notebook and its dataset are released under Apache 2.0.

In [ ]:
P.write_zip(SUB, WORK / 'submission.zip')